In [0]:
# --------------------------------------------------------------
# Set environment variables for the pipeline run
# --------------------------------------------------------------
# These values are retrieved from the master pipeline task 
# (superstore_pipeline_master_run_id_init) using dbutils.jobs.taskValues.
# They ensure that the ETL, Mart, Features, KPI, and Dashboard code
# runs in the correct environment (dev/qa/prod) without hardcoding paths.
# SUPERSTORE_ENV, SUPERSTORE_PIPELINE_NAME, and SUPERSTORE_PIPELINE_VERSION
# are set for downstream modules to dynamically resolve catalog/schema.
# --------------------------------------------------------------
import os

env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -------------------------------
# Environment Config
# -------------------------------
import sys
sys.path.append("/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/superstore_shared_utilities")
from superstore_platform_config import get_catalog, get_gold_schema, get_mart_schema, get_semantic_layer_schema

# Dynamically get environment-aware catalog and schemas
catalog = get_catalog()            # e.g., superstore_catalog
gold_schema = get_gold_schema()    # e.g., dev_gold
semantic_layer_schema = get_semantic_layer_schema()
mart_schema = get_mart_schema()  

# -------------------------------
# Create kpi View (Semantic Layer for Analysts)
# -------------------------------
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.{semantic_layer_schema}.metrics_daily_kpi AS
SELECT
  order_date,
  SUM(total_sales) AS total_sales,
  SUM(total_profit) AS total_profit,
  ROUND(SUM(total_profit)/SUM(total_sales),4) AS profit_margin
FROM {catalog}.{mart_schema}.mart_sales_daily
GROUP BY order_date
ORDER BY order_date;
""")
